# Notebook 09 — Echo-Based CZ Construction

This notebook adds an **echo-style control sequence** designed to suppress unwanted single-particle dynamics while retaining interaction-induced phase.

The goal is to move beyond:
- blockade alone,
- pulse timing alone,

and toward a more gate-like structure by combining:
- pulse segments,
- axis flips,
- a symmetric echo pattern.

We compare:
- the earlier minimal pulse sequence,
- an echo-style sequence,

using the same effective-unitary diagnostics:
- raw process fidelity,
- phase-corrected fidelity,
- entangling phase,
- leakage.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'scipy', 'matplotlib']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, qeye, tensor, sigmax, sigmay, sesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
sy = sigmay()
n = r * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]
basis_labels = ['|gg>', '|gr>', '|rg>', '|rr>']

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
sy1 = tensor(sy, I)
sy2 = tensor(I, sy)
n1 = tensor(n, I)
n2 = tensor(I, n)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Hamiltonian helpers

In [ ]:
def two_atom_hamiltonian(omega1: float, omega2: float, delta: float, V: float, axis1: str = 'x', axis2: str = 'x'):
    op1 = sx1 if axis1 == 'x' else sy1
    op2 = sx2 if axis2 == 'x' else sy2
    drive = 0.5 * omega1 * op1 + 0.5 * omega2 * op2
    detuning = -delta * (n1 + n2)
    interaction = V * n1 * n2
    return drive + detuning + interaction

def run_segment(state, omega1: float, omega2: float, delta: float, V: float,
                duration: float, axis1: str = 'x', axis2: str = 'x', n_steps: int = 120):
    H = two_atom_hamiltonian(omega1=omega1, omega2=omega2, delta=delta, V=V, axis1=axis1, axis2=axis2)
    times = np.linspace(0.0, duration, n_steps)
    result = sesolve(H, state, times)
    return result.states[-1]

def state_to_column(psi):
    return np.array([basis_state.overlap(psi) for basis_state in basis_states], dtype=complex)


## Sequence definitions

In [ ]:
def run_minimal_sequence(psi0, omega: float, delta: float, V: float,
                         t_pi: float, t_mid: float):
    state1 = run_segment(psi0, omega1=omega, omega2=0.0, delta=delta, V=V, duration=t_pi, axis1='x', axis2='x')
    state2 = run_segment(state1, omega1=0.0, omega2=omega, delta=delta, V=V, duration=t_mid, axis1='x', axis2='y')
    state3 = run_segment(state2, omega1=omega, omega2=0.0, delta=delta, V=V, duration=t_pi, axis1='x', axis2='x')
    return state3

def run_echo_sequence(psi0, omega: float, delta: float, V: float,
                      t_half: float, t_mid: float):
    # Pulse atom 1
    s1 = run_segment(psi0, omega1=omega, omega2=0.0, delta=delta, V=V, duration=t_half, axis1='x', axis2='x')
    # Pulse atom 2
    s2 = run_segment(s1, omega1=0.0, omega2=omega, delta=delta, V=V, duration=t_half, axis1='x', axis2='y')
    # Interaction-heavy middle
    s3 = run_segment(s2, omega1=0.0, omega2=omega, delta=delta, V=V, duration=t_mid, axis1='y', axis2='y')
    # Echo pulse atom 2 with flipped axis
    s4 = run_segment(s3, omega1=0.0, omega2=omega, delta=delta, V=V, duration=t_half, axis1='x', axis2='x')
    # Echo pulse atom 1 with flipped axis
    s5 = run_segment(s4, omega1=omega, omega2=0.0, delta=delta, V=V, duration=t_half, axis1='y', axis2='x')
    return s5

def build_effective_unitary(sequence_fn, omega: float, delta: float, V: float, **kwargs):
    columns = []
    for psi0 in basis_states:
        psi_final = sequence_fn(psi0, omega=omega, delta=delta, V=V, **kwargs)
        columns.append(state_to_column(psi_final))
    return np.column_stack(columns)


## Gate diagnostics

In [ ]:
def process_fidelity(U_eff, U_target=U_cz):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def phase_corrected_target(U_eff):
    phi_ent = entangling_phase(U_eff)
    return np.diag([1, 1, 1, np.exp(1j * phi_ent)]).astype(complex)

def phase_corrected_fidelity(U_eff):
    return process_fidelity(U_eff, U_target=phase_corrected_target(U_eff))

def leakage_metric(U_eff):
    offdiag = U_eff.copy()
    np.fill_diagonal(offdiag, 0)
    return float(np.linalg.norm(offdiag))


## Compare minimal vs echo at one operating point

In [ ]:
omega = 1.0
delta = 0.0
V = 4.0
t_pi = np.pi / omega
t_half = 0.5 * np.pi / omega
t_mid = 2 * np.pi / omega

U_min = build_effective_unitary(run_minimal_sequence, omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid)
U_echo = build_effective_unitary(run_echo_sequence, omega=omega, delta=delta, V=V, t_half=t_half, t_mid=t_mid)

for label, U_eff in [('minimal', U_min), ('echo', U_echo)]:
    print(label)
    print(f'  raw fidelity:             {process_fidelity(U_eff):.6f}')
    print(f'  phase-corrected fidelity: {phase_corrected_fidelity(U_eff):.6f}')
    print(f'  entangling phase / pi:    {entangling_phase(U_eff)/np.pi:.6f}')
    print(f'  leakage:                  {leakage_metric(U_eff):.6f}')
    print()


## Effective-unitary comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
im0 = axes[0].imshow(np.abs(U_min), origin='lower', aspect='equal')
axes[0].set_title(r'$|U_{eff}|$ minimal')
axes[0].set_xticks(range(4), basis_labels)
axes[0].set_yticks(range(4), basis_labels)

im1 = axes[1].imshow(np.abs(U_echo), origin='lower', aspect='equal')
axes[1].set_title(r'$|U_{eff}|$ echo')
axes[1].set_xticks(range(4), basis_labels)
axes[1].set_yticks(range(4), basis_labels)

fig.colorbar(im1, ax=axes.ravel().tolist(), label='magnitude')
plt.tight_layout()
plt.show()


## Scan the middle interaction time

In [ ]:
t_mid_vals = np.linspace(0.6 * np.pi / omega, 3.2 * np.pi / omega, 50)

raw_min, raw_echo = [], []
pc_min, pc_echo = [], []
phi_min, phi_echo = [], []
leak_min, leak_echo = [], []

for t_mid_test in t_mid_vals:
    U_m = build_effective_unitary(run_minimal_sequence, omega=omega, delta=delta, V=V, t_pi=t_pi, t_mid=t_mid_test)
    U_e = build_effective_unitary(run_echo_sequence, omega=omega, delta=delta, V=V, t_half=t_half, t_mid=t_mid_test)

    raw_min.append(process_fidelity(U_m))
    raw_echo.append(process_fidelity(U_e))
    pc_min.append(phase_corrected_fidelity(U_m))
    pc_echo.append(phase_corrected_fidelity(U_e))
    phi_min.append(entangling_phase(U_m) / np.pi)
    phi_echo.append(entangling_phase(U_e) / np.pi)
    leak_min.append(leakage_metric(U_m))
    leak_echo.append(leakage_metric(U_e))


In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, raw_min, label='raw minimal')
plt.plot(t_mid_vals, raw_echo, label='raw echo')
plt.axvline(2 * np.pi / omega, linestyle='--', label=r'$2\pi/\Omega$')
plt.xlabel(r'Middle interaction time $t_{mid}$')
plt.ylabel('Raw process fidelity')
plt.title('Raw fidelity: minimal vs echo')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, pc_min, label='phase-corrected minimal')
plt.plot(t_mid_vals, pc_echo, label='phase-corrected echo')
plt.axvline(2 * np.pi / omega, linestyle='--', label=r'$2\pi/\Omega$')
plt.xlabel(r'Middle interaction time $t_{mid}$')
plt.ylabel('Phase-corrected fidelity')
plt.title('Phase-corrected fidelity: minimal vs echo')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, phi_min, label='minimal')
plt.plot(t_mid_vals, phi_echo, label='echo')
plt.axhline(1.0, linestyle='--', label='ideal +1')
plt.axhline(-1.0, linestyle='--', color='gray', label='ideal -1')
plt.xlabel(r'Middle interaction time $t_{mid}$')
plt.ylabel(r'Entangling phase / $\pi$')
plt.title('Entangling phase: minimal vs echo')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 4.8))
plt.plot(t_mid_vals, leak_min, label='minimal')
plt.plot(t_mid_vals, leak_echo, label='echo')
plt.xlabel(r'Middle interaction time $t_{mid}$')
plt.ylabel('Leakage norm')
plt.title('Leakage: minimal vs echo')
plt.legend()
plt.tight_layout()
plt.show()


## Echo-sequence optimization map over $(\Omega, V)$

In [ ]:
omega_grid = np.linspace(0.6, 1.8, 22)
V_grid = np.linspace(0.5, 6.0, 22)

raw_map = np.zeros((len(V_grid), len(omega_grid)))
pc_map = np.zeros((len(V_grid), len(omega_grid)))
phi_map = np.zeros((len(V_grid), len(omega_grid)))
leak_map = np.zeros((len(V_grid), len(omega_grid)))

for i, V_scan in enumerate(V_grid):
    for j, omega_scan in enumerate(omega_grid):
        t_half_scan = 0.5 * np.pi / omega_scan
        t_mid_scan = 2 * np.pi / omega_scan
        U_eff = build_effective_unitary(
            run_echo_sequence,
            omega=omega_scan,
            delta=0.0,
            V=V_scan,
            t_half=t_half_scan,
            t_mid=t_mid_scan
        )
        raw_map[i, j] = process_fidelity(U_eff)
        pc_map[i, j] = phase_corrected_fidelity(U_eff)
        phi_map[i, j] = entangling_phase(U_eff) / np.pi
        leak_map[i, j] = leakage_metric(U_eff)


In [ ]:
plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(raw_map, origin='lower', aspect='auto', extent=[omega_grid[0], omega_grid[-1], V_grid[0], V_grid[-1]])
plt.contour(omega_grid, V_grid, raw_map, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Raw process fidelity over $(\Omega, V)$ for echo sequence')
plt.colorbar(im, label='raw fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(pc_map, origin='lower', aspect='auto', extent=[omega_grid[0], omega_grid[-1], V_grid[0], V_grid[-1]])
plt.contour(omega_grid, V_grid, pc_map, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Phase-corrected fidelity over $(\Omega, V)$ for echo sequence')
plt.colorbar(im, label='phase-corrected fidelity')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(phi_map, origin='lower', aspect='auto', extent=[omega_grid[0], omega_grid[-1], V_grid[0], V_grid[-1]], vmin=-1, vmax=1)
plt.contour(omega_grid, V_grid, phi_map, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Entangling phase / $\pi$ over $(\Omega, V)$ for echo sequence')
plt.colorbar(im, label=r'$\phi_{ent}/\pi$')
plt.tight_layout()
plt.show()

plt.figure(figsize=(7.5, 5.5))
im = plt.imshow(leak_map, origin='lower', aspect='auto', extent=[omega_grid[0], omega_grid[-1], V_grid[0], V_grid[-1]])
plt.contour(omega_grid, V_grid, leak_map, colors='white', linewidths=0.4)
plt.xlabel(r'Rabi frequency $\Omega$')
plt.ylabel(r'Interaction strength $V$')
plt.title(r'Leakage over $(\Omega, V)$ for echo sequence')
plt.colorbar(im, label='leakage norm')
plt.tight_layout()
plt.show()


## Interpretation

This notebook introduces a more symmetric, echo-style control structure.

The main question is whether echo symmetry can improve gate quality by:
- reducing single-particle phase accumulation,
- preserving the entangling phase,
- lowering leakage.

The most useful comparison is not just raw fidelity, but whether the echo sequence improves:
- phase-corrected fidelity,
- entangling phase stability,
- computational-basis return.


## Optional next steps

Natural next upgrades are:

- add detuning ramps in the interaction segment,
- scan over pulse-axis choices more broadly,
- add explicit local phase corrections after the echo sequence,
- replace the hand-designed sequence with numerical optimal control.
